In [1]:
# NOTEBOOK 38 - forecast WEATHER for the 24 future hours: fetch it, check its quality, and test whether it is worth training on (no neural model)
# Idea: forecast wind was the one input that clearly improved the score. Weather (temperature, humidity, cloud, pressure, rain, sunshine) controls mixing, fog and
# washout, but the decoder only ever sees past weather. This notebook gets the day-old forecast of those six variables (Open-Meteo Previous Runs API, the same
# source and lead as the wind forecast in notebook 18) for the 37 stations and answers three questions before any training:
#   1) is the data complete?   2) is the forecast any good (against what the stations measured)?   3) does it carry information about the next 24 h of PM2.5
#      beyond what the model already knows?  Question 3 uses a plain linear model with the day-old forecast WIND as the calibration (we know wind helps).
# Writes new files only: openmeteo_prevday1_weather.npy, openmeteo_prevday1_weather_mask.npy, openmeteo_prevday1_weather_meta.json (v2_clean folder).
# Takes about 10 minutes (the download). Boundary-layer height and visibility are NOT available in this API (checked), so they are not used.
# HOW TO CHECK: this cell only imports and sets paths; it prints N == 37, T == 19921.
import os, time, json
import numpy as np
import pandas as pd
import requests

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
DATASET_CSV    = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')

df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
lats = df_st['latitude'].values.astype(float); lons = df_st['longitude'].values.astype(float)
N = len(station_names)
ds = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp']); ds['Station'] = ds['Station'].str.strip()
timestamps = pd.DatetimeIndex(sorted(ds['Timestamp'].unique())); T = len(timestamps)
START, END = timestamps[0], timestamps[-1]
print(f'Stations {N} | grid {T:,} hourly steps {START} -> {END}')

Stations 37 | grid 19,921 hourly steps 2024-01-01 00:00:00 -> 2026-04-10 00:00:00


In [2]:
# fetch helpers (same pattern as notebook 18, six variables per request)
URL = 'https://previous-runs-api.open-meteo.com/v1/forecast'
WX = ['temperature_2m', 'relative_humidity_2m', 'surface_pressure', 'precipitation', 'cloud_cover', 'shortwave_radiation']
CHUNK_DAYS = 120
def fetch_chunk(lat, lon, d0, d1, retries=5):
    params = {'latitude': lat, 'longitude': lon, 'start_date': d0.strftime('%Y-%m-%d'), 'end_date': d1.strftime('%Y-%m-%d'),
              'hourly': ','.join(v + '_previous_day1' for v in WX), 'timezone': 'Asia/Kolkata'}
    for k in range(retries):
        try:
            r = requests.get(URL, params=params, timeout=90)
            if r.status_code == 200:
                h = r.json()['hourly']
                return pd.DataFrame({v: pd.to_numeric(pd.Series(h[v + '_previous_day1']), errors='coerce').values for v in WX}, index=pd.DatetimeIndex(h['time']))
            if r.status_code in (429, 503): time.sleep(10 * (k + 1)); continue
            raise RuntimeError(f'HTTP {r.status_code}: {r.text[:200]}')
        except (requests.Timeout, requests.ConnectionError):
            time.sleep(5 * (k + 1))
    raise RuntimeError('failed after retries')
def fetch_station(lat, lon):
    frames, d0, last = [], START.normalize(), END.normalize()
    while d0 <= last:
        d1 = min(d0 + pd.Timedelta(days=CHUNK_DAYS - 1), last)
        frames.append(fetch_chunk(lat, lon, d0, d1)); d0 = d1 + pd.Timedelta(days=1); time.sleep(0.6)
    s = pd.concat(frames); return s[~s.index.duplicated()]
print('helpers ready; variables:', WX)

helpers ready; variables: ['temperature_2m', 'relative_humidity_2m', 'surface_pressure', 'precipitation', 'cloud_cover', 'shortwave_radiation']


In [3]:
# 1) download for all 37 stations (about 8-10 minutes). The API has no day-old forecasts before about 19 Jan 2024: short gaps (3 h or less) are interpolated and the
#    rest is filled with that station's average for the hour of the day over its first 28 real days, and FLAGGED in a mask (the first weeks of the training period only).
FW = np.full((T, N, len(WX)), np.nan, dtype=np.float32); filled = np.zeros((T, N), dtype=bool); t0 = time.time()
for i, (nm, la, lo) in enumerate(zip(station_names, lats, lons)):
    s = fetch_station(la, lo).reindex(timestamps)
    s = s.interpolate(limit=3, limit_area='inside')
    miss = s.isna().any(axis=1).values
    real0 = np.flatnonzero(~miss)[0]; ref = s.iloc[real0: real0 + 28 * 24]
    prof = ref.groupby(ref.index.hour).mean()
    for v in WX: s[v] = np.where(s[v].isna(), s.index.hour.map(prof[v]), s[v])
    FW[:, i, :] = s[WX].values.astype(np.float32); filled[:, i] = miss
    if (i + 1) % 10 == 0 or i == N - 1: print(f'  [{i + 1:>2}/{N}] {nm[:24]:<24} {time.time() - t0:>4.0f}s')
print(f'\nshape {FW.shape} | NaN {int(np.isnan(FW).sum())} | filled {filled.mean() * 100:.1f}% of hours | first fully-real hour {timestamps[np.argmax(~filled.any(1))]}')
print('mean of each variable:', {v: round(float(FW[..., k].mean()), 2) for k, v in enumerate(WX)})
print('per-variable stations that look identical (same grid cell):', int(len(np.unique(FW[:, :, 0].round(2), axis=1).T)), 'distinct temperature series out of', N)

# HOW TO CHECK: "filled" about 2% (only Jan 2024). If much larger, stop and tell me. Distinct series is smaller than 37 because several stations share a forecast grid cell.

  [10/37] Dr. Karni Singh Shooting  151s
  [20/37] Okhla Phase-2             280s
  [30/37] Burari Crossing           401s
  [37/37] Wazirpur                  491s

shape (19921, 37, 6) | NaN 0 | filled 2.3% of hours | first fully-real hour 2024-01-19 17:00:00
mean of each variable: {'temperature_2m': 24.09, 'relative_humidity_2m': 65.2, 'surface_pressure': 983.78, 'precipitation': 0.12, 'cloud_cover': 43.33, 'shortwave_radiation': 202.75}
per-variable stations that look identical (same grid cell): 32 distinct temperature series out of 37


In [4]:
# 2) save (new files only)
np.save(os.path.join(V2_DIR, 'openmeteo_prevday1_weather.npy'), FW)
np.save(os.path.join(V2_DIR, 'openmeteo_prevday1_weather_mask.npy'), filled)
json.dump({'source': 'previous-runs-api.open-meteo.com, *_previous_day1', 'variables': WX, 'units': 'C, %, hPa, mm, %, W/m2', 'lead': 'about 1 day',
           'start': str(START), 'end': str(END), 'timezone': 'Asia/Kolkata', 'filled_fraction': float(filled.mean())},
          open(os.path.join(V2_DIR, 'openmeteo_prevday1_weather_meta.json'), 'w'), indent=2)
print('saved openmeteo_prevday1_weather.npy', FW.shape, '| mask + meta saved')

saved openmeteo_prevday1_weather.npy (19921, 37, 6) | mask + meta saved


In [5]:
# 3) QUALITY: how close is the day-old forecast to what the stations measured, and is it better than just repeating yesterday?
#    Only temperature and humidity have a station measurement in the dataset (these were partly imputed, so the comparison is a little optimistic or pessimistic).
obs = {v: ds.pivot_table(index='Timestamp', columns='Station', values=c).reindex(index=timestamps, columns=station_names).values for v, c in (('temperature_2m', 'at'), ('relative_humidity_2m', 'rh'))}
cut = int(T * 0.85); real = ~filled
print(f"{'variable':22}{'corr':>7}{'bias':>8}{'RMSE':>8}{'RMSE of repeating yesterday':>30}   (all real hours | test period only)")
for v in obs:
    k = WX.index(v); f, o = FW[:, :, k], obs[v]
    ok = real & np.isfinite(o); y = np.roll(o, 24, axis=0); okp = ok & np.roll(np.isfinite(o), 24, axis=0); okp[:24] = False
    c = np.corrcoef(f[ok], o[ok])[0, 1]; b = float((f[ok] - o[ok]).mean()); r = float(np.sqrt(np.mean((f[ok] - o[ok]) ** 2))); ry = float(np.sqrt(np.mean((y[okp] - o[okp]) ** 2)))
    okt = ok.copy(); okt[:cut] = False; okpt = okp.copy(); okpt[:cut] = False
    rt = float(np.sqrt(np.mean((f[okt] - o[okt]) ** 2))); ryt = float(np.sqrt(np.mean((y[okpt] - o[okpt]) ** 2)))
    print(f'{v:22}{c:>7.2f}{b:>+8.1f}{r:>8.1f}{ry:>30.1f}   test: RMSE {rt:.1f} vs yesterday {ryt:.1f}')
print()
print('other variables, no station measurement to compare with - basic range check:')
for v in ('surface_pressure', 'precipitation', 'cloud_cover', 'shortwave_radiation'):
    a = FW[:, :, WX.index(v)]; print(f'  {v:20} mean {a.mean():8.2f}  p5 {np.percentile(a, 5):8.2f}  p95 {np.percentile(a, 95):8.2f}  share of hours above 0: {(a > 0.05).mean() * 100:.0f}%')

# HOW TO READ: a useful forecast has a correlation above about 0.8 for temperature (humidity 0.6-0.8) and an RMSE clearly BELOW "repeating yesterday".
# A large systematic bias (more than a few degrees or 10 percentage points) is fine - the model learns a constant offset - as long as the correlation is high.

variable                 corr    bias    RMSE   RMSE of repeating yesterday   (all real hours | test period only)
temperature_2m           0.90    -0.7     3.6                           2.0   test: RMSE 3.5 vs yesterday 1.7
relative_humidity_2m     0.82    +0.9    13.0                           9.1   test: RMSE 15.2 vs yesterday 10.4

other variables, no station measurement to compare with - basic range check:
  surface_pressure     mean   983.78  p5   972.30  p95   993.90  share of hours above 0: 100%
  precipitation        mean     0.12  p5     0.00  p95     0.70  share of hours above 0: 14%
  cloud_cover          mean    43.33  p5     0.00  p95   100.00  share of hours above 0: 64%
  shortwave_radiation  mean   202.75  p5     0.00  p95   774.00  share of hours above 0: 53%


In [6]:
# 4) RELEVANCE: does the forecast of the next 24 h of weather tell us anything about the next 24 h of PM2.5 that the model does not already know?
#    Target: change of the city-mean log PM2.5 from now to the average of the next 24 h. Winter months only (Oct-Mar).
#    Controls (what the model already sees): Delhi level, its 6 h and 24 h change, hour of day, month, and the CURRENT city-mean temperature, humidity and wind speed.
#    Features: for each variable, (forecast mean over the next 24 h) and (forecast mean over the next 24 h minus the same forecast product over the past 24 h).
#    Model: plain linear regression trained on winter hours before 8 Dec 2025, scored on winter hours after it (the Delhi test period). The gain in R2 is what it is worth.
#    The day-old forecast WIND is run the same way as a calibration: we know forecast wind helps, so weather needs to look at least comparable to be promising.
logp = np.log1p(ds.pivot_table(index='Timestamp', columns='Station', values='pm2.5').reindex(index=timestamps, columns=station_names).values)
DEL = logp.mean(axis=1)
era5 = np.load(os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')).astype(np.float32); pv = np.load(os.path.join(V2_DIR, 'openmeteo_prevday1_uv.npy')).astype(np.float32)
cur_t = np.nanmean(obs['temperature_2m'], axis=1); cur_rh = np.nanmean(obs['relative_humidity_2m'], axis=1); cur_ws = np.hypot(era5[..., 0], era5[..., 1]).mean(axis=1)
city_fc = {v: FW[:, :, k].mean(axis=1) for k, v in enumerate(WX)}; city_fc['wind_speed'] = np.hypot(pv[..., 0], pv[..., 1]).mean(axis=1)
H = 24; t = np.arange(48, T - H)
cs = lambda a: np.concatenate([[0], np.cumsum(a)])
def next_mean(a): c = cs(a); return (c[t + H + 1] - c[t + 1]) / H            # mean of a[t+1 .. t+24]
def past_mean(a): c = cs(a); return (c[t + 1] - c[t - H + 1]) / H            # mean of a[t-23 .. t]
tgt = next_mean(DEL) - DEL[t]
hr = np.eye(24)[timestamps.hour.values[t]][:, 1:]; mo = np.column_stack([np.sin(2 * np.pi * (timestamps.month.values[t] - 1) / 12), np.cos(2 * np.pi * (timestamps.month.values[t] - 1) / 12)])
ctrl = np.column_stack([DEL[t], DEL[t] - DEL[t - 6], DEL[t] - DEL[t - 24], cur_t[t], cur_rh[t], cur_ws[t], hr, mo])
def feats(names): return np.column_stack([f(city_fc[v]) for v in names for f in (next_mean, lambda a: next_mean(a) - past_mean(a))])
WEATHER = list(WX); WIND = ['wind_speed']
winter = np.isin(timestamps.month.values[t], [10, 11, 12, 1, 2, 3]); SPLIT = int(np.searchsorted(timestamps, pd.Timestamp('2025-12-08')))
base_ok = np.isfinite(tgt) & np.isfinite(ctrl).all(1) & winter
tr, te = base_ok & (t < SPLIT - H), base_ok & (t >= SPLIT)
def fit_score(extra):
    X = np.column_stack([np.ones(len(t)), ctrl] + ([extra] if extra is not None else []))
    mu, sd = X[tr].mean(0), X[tr].std(0); sd[sd == 0] = 1; mu[0], sd[0] = 0, 1
    Z = (X - mu) / sd; w = np.linalg.lstsq(Z[tr], tgt[tr], rcond=None)[0]; p = Z[te] @ w
    return 1 - np.sum((tgt[te] - p) ** 2) / np.sum((tgt[te] - tgt[tr].mean()) ** 2), float(np.sqrt(np.mean((tgt[te] - p) ** 2)))
rows = [('controls only (what the model already knows)', None), ('+ forecast wind', feats(WIND)), ('+ forecast weather (6 variables)', feats(WEATHER)),
        ('+ forecast wind + weather', feats(WIND + WEATHER))] + [(f'+ forecast {v} only', feats([v])) for v in WEATHER]
print(f'winter hours: train {int(tr.sum())}, test {int(te.sum())} (test = after 8 Dec 2025)')
print(f"{'features':46}{'test R2':>9}{'gain':>8}{'RMSE (log units)':>18}")
r0 = fit_score(None)
for nm, ex in rows:
    r2_, rm = fit_score(ex); print(f'{nm:46}{r2_:>9.3f}{r2_ - r0[0]:>+8.3f}{rm:>18.4f}')

# HOW TO READ: this linear model is a crude detector, but it ranks the options. If "+ forecast weather" gains roughly as much as "+ forecast wind" (or at least about
# +0.01 R2), training a model on it is worth the 50 minutes. If the gain is about 0.00-0.003, weather forecasts add nothing the model does not already infer - stop here.

winter hours: train 8112, test 2736 (test = after 8 Dec 2025)
features                                        test R2    gain  RMSE (log units)
controls only (what the model already knows)      0.614  +0.000            0.2319
+ forecast wind                                   0.639  +0.025            0.2243
+ forecast weather (6 variables)                  0.610  -0.004            0.2332
+ forecast wind + weather                         0.634  +0.020            0.2259
+ forecast temperature_2m only                    0.627  +0.012            0.2282
+ forecast relative_humidity_2m only              0.611  -0.003            0.2329
+ forecast surface_pressure only                  0.604  -0.010            0.2350
+ forecast precipitation only                     0.606  -0.009            0.2345
+ forecast cloud_cover only                       0.614  +0.000            0.2319
+ forecast shortwave_radiation only               0.614  -0.000            0.2320
